In [ ]:
import logging
import sys
import re

from typing import List, Dict, Optional
from datetime import datetime
from bs4 import BeautifulSoup
from pydantic import ValidationError
from bs4.element import Tag

In [ ]:
html = '../data/raw/html/2026_03_29-23h_19m/pagina_161.html'

#Patterns 
SPAN_PRICE_PATTERN = re.compile(".*amount__large__price.*")
FOOTER_PATTERN = re.compile(".*product_cardProduct__footerInfo.*")
SUBTITLE_PATTERN = re.compile(".*Product__subtitle.*")
BANNER_PATTERN = re.compile("Apartado")


In [ ]:
def read_html(file: str) -> BeautifulSoup:
    """Abre un archivo HTML y retorna un objeto de BeautifulSoup"""
    with open(file, 'r', encoding='utf-8') as f:
        html = BeautifulSoup(f, 'html.parser')
    return html

In [ ]:
beauty_html = read_html(html)

In [ ]:
def get_cards(html: BeautifulSoup) -> List[Tag]:
    """Encuentra las etiquetas <a> dentro del HTML donde el attribute 'data-testid' contenga 'card-product' dentro."""
    return [
        tag for tag in html.find_all('a')
        if tag.get('data-testid') and 'card-product' in tag.get('data-testid')
    ]


In [ ]:
cards = get_cards(beauty_html) 

In [ ]:
print(cards[1])

In [ ]:
car_id = cards[1]['data-testid'].replace('-', ' ').split()[-1]

In [ ]:
def extract_price(card: Tag, car_id: str) -> Optional[int]:
    """Encontramos el precio del vehiculo en la card, si no tiene continua con la siguiente iteracion."""
    price_span = card.find(class_=SPAN_PRICE_PATTERN)
    print(price_span)
    if price_span:
        try:
            price_text = price_span.get_text(strip=True).replace(',', '').replace('$', '')
            price = int(price_text)
            return price
        except (ValueError, AttributeError) as e:
            print(e)
    return None

def extract_banner(card: Tag) -> int:
    """Extraccion de banners"""
    hot_sale_flag = 0
    banner = card.find(string=BANNER_PATTERN)
    if banner:
        return banner
    return None

In [ ]:
banner = extract_banner(cards[1])
price = extract_price(cards[1], car_id)

In [ ]:
print(banner)
print(price)